# RHI Live Runtime v7
## Shape-First Binary Stance Gate + Shape Training Rows

This fixes the v6 tear:

```text
v6 gap — binary stance prompts could bypass VERIFY by barely clearing margin
example: "filter or gate?" collapsed by margin before stance divergence was checked
```

v7 adds the missing abstraction layer:

```text
FILTER   = excludes / removes / screens invalid states
GATE     = conditionally permits or blocks transition
LOCK     = prevents transition until key/condition
CONTRACT = binds future action to prior conditions
BOUNDARY = defines valid interface crossing
GROOVE   = lowers resistance along a preferred path
RESIDUE  = unresolved mismatch left after collapse
```

Runtime:

```text
prompt
  ↓
slot_builder_lora_v2 emits raw contract
  ↓
contract repair gate fixes polarity and scar leakage
  ↓
base model generates answer branches
  ↓
deterministic five-dimensional critic scores branches
  ↓
shape stance extractor abstracts answers into operation-shapes
  ↓
KRRB:
      if binary stance conflict → VERIFY shape stance before margin collapse
      else normal GATE / VOTE / VERIFY
      else Ω
```

Output:

```text
rhi_live_runtime_v7_outputs/
  rhi_live_runs_v7.jsonl
  rhi_repair_training_rows_v7.jsonl
  rhi_shape_training_rows_v7.jsonl
  rhi_live_runtime_v7_manifest.json
```


In [1]:
# ============================================================
# CONFIG
# ============================================================
from pathlib import Path

ROOT             = Path.cwd()
MODEL_NAME       = "Qwen/Qwen2.5-1.5B-Instruct"
SLOT_ADAPTER_DIR = ROOT / "slot_builder_lora_v2"

OUTPUT_DIR = ROOT / "rhi_live_runtime_v7_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_FILES_ONLY = True
USE_4BIT         = False

CONTRACT_MAX_NEW_TOKENS = 700
ANSWER_MAX_NEW_TOKENS   = 900
RESIDUE_MAX_NEW_TOKENS  = 500

N_BRANCHES             = 5
DO_SAMPLE_FOR_BRANCHES = True
BRANCH_TEMPERATURE     = 0.55
BRANCH_TOP_P           = 0.92

# Collapse thresholds (unchanged from v5).
SUPPORT_MIN = 4
MARGIN_MIN  = 0.04
PSI_MIN     = 0.52
AUDIT_MIN   = 0.52

# Consensus gates.
CONSENSUS_MARGIN_MAX    = 0.04
CONSENSUS_AGREEMENT_MIN = 0.36
CONSENSUS_AUDIT_MIN     = 0.86
CONSENSUS_STANCE_MIN    = 0.50

ADD_MODEL_RESIDUE_COMMENTARY = True

# v7: shape-first binary conflict controls.
SHAPE_STANCE_CONF_MIN = 0.18
SHAPE_OPPONENT_QUALITY_MIN = 0.52
SHAPE_VERIFY_TOP_N = 4

# v7: training signal export.
EXPORT_TRAINING_SIGNAL   = True
TRAINING_SIGNAL_FILE     = OUTPUT_DIR / "rhi_repair_training_rows_v7.jsonl"
SHAPE_SIGNAL_FILE        = OUTPUT_DIR / "rhi_shape_training_rows_v7.jsonl"

print("ROOT:", ROOT)
print("SLOT_ADAPTER_DIR:", SLOT_ADAPTER_DIR, SLOT_ADAPTER_DIR.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)


ROOT: D:\@User Data\Downloads
SLOT_ADAPTER_DIR: D:\@User Data\Downloads\slot_builder_lora_v2 True
OUTPUT_DIR: D:\@User Data\Downloads\rhi_live_runtime_v7_outputs


In [2]:
# ============================================================
# IMPORTS
# ============================================================
INSTALL_MISSING = False
if INSTALL_MISSING:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U",
        "transformers", "peft", "accelerate", "sentencepiece", "pandas"])

import json, re, time, uuid
from datetime import datetime
from typing import Any, Dict, List, Optional, Tuple

import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("vram GB:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))


torch: 2.11.0+cu126
cuda: True
gpu: NVIDIA GeForce RTX 4060
vram GB: 8.0


In [3]:
# ============================================================
# HELPERS
# ============================================================
REQUIRED_FIELDS = [
    "family_class", "domain_carrier", "forbidden_neighbor_carrier",
    "boundary_conditions", "preserved_function", "failure_modes",
    "witness_readout", "residue",
]
AUDIT_FIELDS = ["F_need", "F_function", "F_boundary", "F_trap", "F_collapse"]

STOPWORDS = {
    "the","a","an","and","or","of","to","in","on","for","with","as","is","are",
    "was","were","be","being","been","it","its","this","that","these","those",
    "by","from","into","at","while","what","when","where","why","how","which",
    "who","whom","one","two","three","do","does","did","not","no","yes","can",
    "could","should","would","will","may","might","using","use","used","uses",
    "current","answer","task","prompt","question","response",
}

NEXUS_VOCAB = {
    "contract","boundary","domain_carrier","forbidden","preserved_function","witness_readout",
    "boundary_conditions","failure_modes","family_class","residue","carrier","collapse",
    "nexus","slot","operational","fit","missing","shape","cavity","rhi","krrb",
}

SCAR_TERMS = {
    "string wraps but does not center under rotation",
    "last paragraph is here",
    "unrelated_to_prompt",
    "wrong neighboring domain carrier",
    "surface label without operational fit",
    "general purpose","purpose","not answered","nonexistent",
    "name-only rubber-part match",
    "permanent membership without local constraint satisfaction",
}

GENERIC_DOMAIN_TERMS = {
    "current","failing","form","use","using","properly","understanding",
    "answer","task","prompt","question","response","chosen","collapse",
}

# v7: abstract operation-shape ontology.
SHAPE_ONTOLOGY = {
    "FILTER": {
        "labels": {"filter", "filters", "filtering"},
        "verbs": {"remove", "removes", "exclude", "excludes", "screen", "screens", "reject", "rejects", "discard", "blocks"},
        "nouns": {"invalid", "noise", "candidate", "state", "states", "set", "selection"},
        "gloss": "excludes or removes invalid states from a candidate set",
    },
    "GATE": {
        "labels": {"gate", "gates", "gating"},
        "verbs": {"allow", "allows", "permit", "permits", "open", "opens", "close", "closes", "blocks", "transition", "cross"},
        "nouns": {"condition", "permission", "transition", "threshold", "entry", "passage", "interface"},
        "gloss": "conditionally permits or blocks transition across a boundary",
    },
    "LOCK": {
        "labels": {"lock", "locks", "locking"},
        "verbs": {"prevent", "prevents", "hold", "holds", "requires", "unlock", "release"},
        "nouns": {"key", "condition", "constraint", "permission", "state"},
        "gloss": "prevents transition until a key or condition is satisfied",
    },
    "CONTRACT": {
        "labels": {"contract", "contracts"},
        "verbs": {"bind", "binds", "commit", "commits", "constrain", "constrains", "specify", "specifies"},
        "nouns": {"intent", "terms", "boundary", "obligation", "precondition", "action"},
        "gloss": "binds future action to prior conditions and intent",
    },
    "BOUNDARY": {
        "labels": {"boundary", "boundaries"},
        "verbs": {"separate", "separates", "define", "defines", "cross", "limits", "constrain"},
        "nouns": {"interface", "edge", "condition", "crossing", "limit", "domain"},
        "gloss": "defines the valid interface crossing between states",
    },
    "GROOVE": {
        "labels": {"groove", "grooves", "grooving"},
        "verbs": {"lower", "lowers", "bias", "biases", "guide", "guides", "adapt", "adapts"},
        "nouns": {"path", "resistance", "adapter", "rank", "manifold", "delta", "lora"},
        "gloss": "lowers resistance along a preferred path without rewriting the whole field",
    },
    "RESIDUE": {
        "labels": {"residue", "repair", "omega", "Ω"},
        "verbs": {"remain", "remains", "repair", "repairs", "backpatch", "capture", "captures"},
        "nouns": {"mismatch", "failure", "trace", "error", "signal", "memory"},
        "gloss": "unresolved mismatch left after collapse, captured for repair",
    },
}

# Dangling prepositions / articles that signal an incomplete family_class phrase.
FRAGMENT_ENDINGS = {"of","for","with","to","in","by","and","or","a","an","the","from","at"}

def now_iso():   return datetime.now().isoformat(timespec="seconds")
def safe_div(a,b): return float(a)/float(b) if b else 0.0
def clamp01(x):
    try: return max(0.0, min(1.0, float(x)))
    except: return 0.0

def wordset(text: Any) -> set:
    if isinstance(text, list): text = " ".join(map(str, text))
    toks = re.findall(r"[a-zA-Z0-9_]+", str(text).lower())
    return {t for t in toks if t not in STOPWORDS and len(t) > 1}

def wordset_no_nexus(text: Any) -> set:
    return wordset(text) - NEXUS_VOCAB

def phrase_present(text: str, phrase: str) -> bool:
    return phrase.lower() in str(text).lower()

def extract_first_json_object(text: str) -> Tuple[Optional[Dict], Optional[str]]:
    text = str(text).strip()
    try:
        obj = json.loads(text)
        return (obj, None) if isinstance(obj, dict) else (None, "json_not_dict")
    except: pass
    cleaned = re.sub(r"^```(?:json)?", "", text, flags=re.IGNORECASE).strip()
    cleaned = re.sub(r"```$", "", cleaned).strip()
    try:
        obj = json.loads(cleaned)
        return (obj, None) if isinstance(obj, dict) else (None, "fenced_json_not_dict")
    except: pass
    start, end = text.find("{"), text.rfind("}")
    if start >= 0 and end > start:
        try:
            obj = json.loads(text[start:end+1])
            return (obj, None) if isinstance(obj, dict) else (None, "scanned_json_not_dict")
        except Exception as e:
            return None, "json_parse_error: " + str(e)
    return None, "no_json_object_found"

def normalize_list(x):
    if x is None: return []
    if isinstance(x, list): return [str(v).strip() for v in x if str(v).strip()]
    if isinstance(x, str):
        s = x.strip()
        if not s: return []
        if s.startswith("{") or ":" in s: return [s]
        return [p.strip() for p in re.split(r"[|,;]", s) if p.strip()]
    return [str(x).strip()]

def normalize_contract(c0: Optional[Dict]) -> Dict:
    c0 = c0 or {}
    return {
        "family_class":               str(c0.get("family_class","") or "").strip(),
        "domain_carrier":             normalize_list(c0.get("domain_carrier",[])),
        "forbidden_neighbor_carrier": normalize_list(c0.get("forbidden_neighbor_carrier",[])),
        "boundary_conditions":        normalize_list(c0.get("boundary_conditions",[])),
        "preserved_function":         str(c0.get("preserved_function","") or "").strip(),
        "failure_modes":              normalize_list(c0.get("failure_modes",[])),
        "witness_readout":            str(c0.get("witness_readout","") or "").strip(),
        "residue":                    c0.get("residue", None),
    }

def contract_complete(c: Optional[Dict]) -> bool:
    if not isinstance(c, dict): return False
    c = normalize_contract(c)
    for f in REQUIRED_FIELDS:
        if f == "residue": continue
        v = c.get(f)
        if isinstance(v, list) and len(v) == 0: return False
        if not isinstance(v, list) and not str(v or "").strip(): return False
    return True

def stable_dedupe(items):
    out, seen = [], set()
    for x in items:
        s = str(x).strip()
        if s and s not in seen:
            seen.add(s); out.append(s)
    return out

# ── v7: family_class fragment repair ─────────────────────────────────────────

def _is_family_class_fragment(fc: str) -> bool:
    """Detect dangling-preposition or too-short family_class values."""
    fc = str(fc or "").strip()
    if not fc:
        return True
    words = fc.lower().split()
    last  = words[-1].rstrip(".,;:") if words else ""
    if last in FRAGMENT_ENDINGS:
        return True
    # Single generic word with no domain content.
    if len(words) == 1 and last in NEXUS_VOCAB | STOPWORDS | {"class","type","kind","group"}:
        return True
    return False

def repair_family_class(
    fc: str,
    preserved_function: str,
    prompt: str,
) -> Tuple[str, Optional[str]]:
    """
    Complete or replace a fragment family_class.
    Strategy (in order):
      1. If fragment ends with preposition, complete it from the most
         distinctive non-NEXUS words in the preserved_function.
      2. If no signal in preserved_function, use the prompt's top content words.
      3. If still nothing, fall back to 'operational closure'.
    Returns (repaired_string, repair_tag) or (fc, None) if no repair needed.
    """
    fc = str(fc or "").strip()
    if not _is_family_class_fragment(fc):
        return fc, None

    # Candidate completion words: function first, then prompt.
    func_words = [w for w in wordset_no_nexus(preserved_function)
                  if w not in STOPWORDS and len(w) > 3]
    prompt_words = [w for w in wordset_no_nexus(prompt)
                    if w not in STOPWORDS and len(w) > 3]

    completion_pool = stable_dedupe(func_words + prompt_words)

    if not fc or fc.lower() in {"", "none", "null"}:
        tag = "family_class_derived"
        if completion_pool:
            return "operational closure: " + " and ".join(completion_pool[:2]), tag
        return "operational closure", tag

    words = fc.lower().split()
    last  = words[-1].rstrip(".,;:")
    if last in FRAGMENT_ENDINGS and completion_pool:
        completed = fc.rstrip() + " " + completion_pool[0]
        return completed, "family_class_fragment_completed"

    # Too short / generic.
    if completion_pool:
        return fc + " [" + completion_pool[0] + "]", "family_class_grounded"

    return fc + " [operational]", "family_class_grounded_default"

# ── v5: contract-anchored stance gate (unchanged) ────────────────────────────

def _field_words(contract: Dict, field: str) -> set:
    val = contract.get(field, [])
    combined = " ".join(val) if isinstance(val, list) else str(val or "")
    return wordset_no_nexus(combined)

def contract_stance_agreement(contract: Dict, answer_a: str, answer_b: str) -> Dict:
    c  = normalize_contract(contract)
    wa = wordset_no_nexus(answer_a)
    wb = wordset_no_nexus(answer_b)
    STANCE_FIELDS = ["domain_carrier","preserved_function","boundary_conditions",
                     "family_class","witness_readout"]
    results = {}
    for field in STANCE_FIELDS:
        fw = _field_words(c, field)
        if not fw:
            results[field] = {"words":[],"overlap_a":1.0,"overlap_b":1.0,"consistency":1.0}
            continue
        ov_a = safe_div(len(wa & fw), len(fw))
        ov_b = safe_div(len(wb & fw), len(fw))
        results[field] = {
            "words": sorted(fw),
            "overlap_a": round(ov_a, 4), "overlap_b": round(ov_b, 4),
            "consistency": round(1.0 - abs(ov_a - ov_b), 4),
        }
    aggregate = sum(v["consistency"] for v in results.values()) / len(results)
    return {"aggregate": round(aggregate, 4), "fields": results,
            "passed": aggregate >= CONSENSUS_STANCE_MIN}

print("helpers ready (family_class repair + stance gate)")


helpers ready (family_class repair + stance gate)


In [4]:
# ============================================================
# LOAD MODEL + SLOT ADAPTER
# ============================================================
if not SLOT_ADAPTER_DIR.exists():
    raise FileNotFoundError("Missing slot adapter folder: " + str(SLOT_ADAPTER_DIR))

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, use_fast=True, local_files_only=LOCAL_FILES_ONLY)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_kwargs = {
    "local_files_only": LOCAL_FILES_ONLY,
    "torch_dtype": torch.float16 if torch.cuda.is_available() else torch.float32,
}
if USE_4BIT:
    from transformers import BitsAndBytesConfig
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    model_kwargs["device_map"] = "auto"

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
if not USE_4BIT and torch.cuda.is_available():
    base_model = base_model.to("cuda")

model  = PeftModel.from_pretrained(base_model, SLOT_ADAPTER_DIR, local_files_only=LOCAL_FILES_ONLY)
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"

print("loaded base:", MODEL_NAME)
print("loaded slot adapter:", SLOT_ADAPTER_DIR)
print("device:", device)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

W0504 12:24:51.808000 14308 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


loaded base: Qwen/Qwen2.5-1.5B-Instruct
loaded slot adapter: D:\@User Data\Downloads\slot_builder_lora_v2
device: cuda


In [5]:
# ============================================================
# GENERATION
# ============================================================
def render_chat(messages: List[Dict[str,str]], add_generation_prompt: bool = False) -> str:
    if hasattr(tokenizer, "apply_chat_template"):
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=add_generation_prompt)
    nl  = chr(10)
    out = [m.get("role","user").upper()+":"+nl+m.get("content","") for m in messages]
    if add_generation_prompt: out.append("ASSISTANT:"+nl)
    return (nl+nl).join(out)

def generate_text(messages, max_new_tokens, do_sample=False,
                  temperature=0.0, top_p=1.0, use_slot_adapter=True) -> str:
    prompt_text = render_chat(messages, add_generation_prompt=True)
    inputs      = tokenizer(prompt_text, return_tensors="pt").to(device)
    gen_kwargs  = {"max_new_tokens": max_new_tokens, "do_sample": do_sample,
                   "pad_token_id": tokenizer.eos_token_id}
    if do_sample:
        gen_kwargs["temperature"] = temperature
        gen_kwargs["top_p"]       = top_p
    with torch.no_grad():
        if use_slot_adapter:
            output_ids = model.generate(**inputs, **gen_kwargs)
        else:
            try:
                with model.disable_adapter():
                    output_ids = model.generate(**inputs, **gen_kwargs)
            except:
                output_ids = model.generate(**inputs, **gen_kwargs)
    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print("generation ready")


generation ready


In [6]:
# ============================================================
# SLOT BUILDER + CONTRACT REPAIR GATE  (v7: +family_class repair)
# ============================================================
SLOT_SYSTEM = "\n".join([
    "You are the Nexus Slot Constructor.",
    "Generate the missing-shape contract before answer selection.",
    "Do not answer the task. Do not mention answer choices.",
    "Return strict JSON only.",
    "Required fields: family_class, domain_carrier, forbidden_neighbor_carrier,",
    "  boundary_conditions, preserved_function, failure_modes, witness_readout, residue.",
    "family_class must be a complete noun phrase naming the operational class.",
    "Use operational fit, not labels.",
])

def build_slot_user_prompt(prompt: str) -> str:
    return "\n".join([
        "Prompt:", prompt, "",
        "Generate the missing-shape contract.",
        "Checklist:",
        "1. Need: occupy the inverse cavity.",
        "2. Function: preserve or redirect the required operation.",
        "3. Boundary: respect constraints.",
        "4. Trap: reject noun/surface-label confusion.",
        "5. Collapse: produce one executable witness/readout.",
        "family_class must be a complete noun phrase (not a dangling preposition).",
        "Return JSON only.",
    ])

def detect_missing_prerequisites(prompt: str) -> Dict:
    p = prompt.lower()
    prereq = {
        "has_before_contract":         bool(re.search(r"before\s+(forming\s+)?a?\s*contract|before\s+contract", p)),
        "has_tool_before_contract":    ("tool" in p or "tools" in p) and ("before" in p and "contract" in p),
        "has_retrieval_before_intent": ("retrieval" in p or "rag" in p) and ("before" in p and "intent" in p),
        "has_filter_or_gate":          ("filter" in p or "gate" in p) and ("boundary" in p or "condition" in p),
    }
    required, forbidden = [], []
    if prereq["has_tool_before_contract"]:
        required.extend(["contract","intent","boundary","sequence","tool use after contract"])
        forbidden.extend(["tool before contract","action without boundary","answer guessing","tool-first action"])
    if prereq["has_retrieval_before_intent"]:
        required.extend(["intent","retrieval","query contract","stabilized need"])
        forbidden.extend(["retrieval before intent","context before contract","semantic drift"])
    prereq["required_terms"]  = stable_dedupe(required)
    prereq["forbidden_terms"] = stable_dedupe(forbidden)
    return prereq

def repair_contract_polarity(contract: Dict, prompt: str) -> Tuple[Dict, List[str]]:
    c      = normalize_contract(contract)
    prereq = detect_missing_prerequisites(prompt)
    repairs = []

    def clean(xs, remove_generic=False):
        out = []
        for x in xs:
            s, low = str(x).strip(), str(x).strip().lower()
            if not s or low in SCAR_TERMS:
                repairs.append("removed_scar:" + s); continue
            if remove_generic and low in GENERIC_DOMAIN_TERMS:
                repairs.append("removed_generic_domain:" + s); continue
            out.append(s)
        return stable_dedupe(out)

    c["domain_carrier"]             = clean(c["domain_carrier"], remove_generic=True)
    c["forbidden_neighbor_carrier"] = clean(c["forbidden_neighbor_carrier"])
    c["failure_modes"]              = clean(c["failure_modes"])

    # ── v7: family_class fragment repair ─────────────────────────────────────
    fc_repaired, fc_tag = repair_family_class(
        c["family_class"], c["preserved_function"], prompt)
    if fc_tag:
        repairs.append(fc_tag)
        c["family_class"] = fc_repaired

    # ── polarity rewrite ──────────────────────────────────────────────────────
    if prereq["has_tool_before_contract"]:
        for term in ["contract","intent","boundary","tool","agent","sequence","tool use after contract"]:
            if term not in c["domain_carrier"]:
                c["domain_carrier"].append(term)
                repairs.append("added_positive:" + term)
        new_forbidden = []
        for term in c["forbidden_neighbor_carrier"]:
            if term.lower() in {"contract","forming contract","forming","intent","boundary"}:
                repairs.append("removed_polarity_inverted_forbidden:" + term); continue
            new_forbidden.append(term)
        c["forbidden_neighbor_carrier"] = new_forbidden
        for term in prereq["forbidden_terms"]:
            if term not in c["forbidden_neighbor_carrier"]:
                c["forbidden_neighbor_carrier"].append(term)
                repairs.append("added_forbidden:" + term)
        c["preserved_function"]  = "form a contract before tool use so action is gated by intent, boundary, and operational fit"
        c["boundary_conditions"] = [
            "contract must be formed before tool use",
            "tool action must be gated by intent and boundary",
            "reject tool-first action, answer guessing, and surface-label routing",
        ]
        c["witness_readout"] = "the answer explains that agents fail when action/tool use occurs before intent, boundary, and contract are stabilized"
        repairs.append("rewrote_boundary_polarity")

    if len(c["domain_carrier"]) < 5:
        for w in sorted(wordset(prompt)):
            if w not in c["domain_carrier"]:
                c["domain_carrier"].append(w)
                repairs.append("added_prompt_carrier:" + w)
            if len(c["domain_carrier"]) >= 7: break

    c["domain_carrier"]             = stable_dedupe(c["domain_carrier"])
    c["forbidden_neighbor_carrier"] = stable_dedupe(c["forbidden_neighbor_carrier"])
    c["failure_modes"]              = stable_dedupe(c["failure_modes"])
    return c, repairs

def generate_contract(prompt: str) -> Dict:
    raw = generate_text(
        [{"role":"system","content":SLOT_SYSTEM},
         {"role":"user",  "content":build_slot_user_prompt(prompt)}],
        max_new_tokens=CONTRACT_MAX_NEW_TOKENS, do_sample=False, use_slot_adapter=True)
    obj, err   = extract_first_json_object(raw)
    original   = normalize_contract(obj) if obj else None
    repaired, repairs = repair_contract_polarity(original, prompt) if original else (None, ["parse_failed"])
    return {
        "raw_contract": raw, "contract_original": original,
        "contract": repaired, "parse_error": err,
        "complete": contract_complete(repaired), "repairs": repairs,
    }

print("slot builder + contract repair gate v7 ready")


slot builder + contract repair gate v7 ready


In [7]:
# ============================================================
# ANSWER BRANCHES
# ============================================================
BRANCH_SYSTEM = "\n".join([
    "You are an answer generator inside an RHI runtime.",
    "Use the provided contract as the operational target.",
    "Answer the user's prompt directly.",
    "Do not output JSON unless the user asked for JSON.",
    "Do not mention internal scoring. Be precise. Do not invent facts.",
])

BRANCH_STYLES = [
    ("direct",        "Answer directly with the clearest useful response.", False, 0.0),
    ("operational",   "Answer by identifying operation, boundary, trap, and witness.", False, 0.0),
    ("contract_fit",  "Answer through the contract: need, preserved function, boundary, and witness.", False, 0.0),
    ("skeptical",     "Reject surface-label traps and explain the failure mode before giving the answer.", True, BRANCH_TEMPERATURE),
    ("residue_aware", "Answer and explicitly flag remaining residue if the contract is incomplete.", True, BRANCH_TEMPERATURE),
]

def branch_user_prompt(prompt: str, contract: Dict, instruction: str) -> str:
    return (
        "User prompt:\n" + prompt
        + "\n\nMissing-shape contract:\n" + json.dumps(normalize_contract(contract), ensure_ascii=False, indent=2)
        + "\n\nBranch instruction:\n" + instruction
        + "\n\nReturn the answer only."
    )

def generate_candidate_branches(prompt: str, contract: Dict) -> List[Dict]:
    branches = []
    for name, instruction, sample, temp in BRANCH_STYLES[:N_BRANCHES]:
        text = generate_text(
            [{"role":"system","content":BRANCH_SYSTEM},
             {"role":"user",  "content":branch_user_prompt(prompt, contract, instruction)}],
            max_new_tokens=ANSWER_MAX_NEW_TOKENS,
            do_sample=bool(sample and DO_SAMPLE_FOR_BRANCHES),
            temperature=float(temp), top_p=BRANCH_TOP_P, use_slot_adapter=False)
        branches.append({"branch": name, "instruction": instruction, "answer": text})
    return branches

print("branches ready")


branches ready


In [8]:
# ============================================================
# DETERMINISTIC FIVE-DIMENSIONAL CRITIC
# ============================================================
def overlap_score(answer_text: str, terms: Any) -> Dict:
    a = wordset(answer_text); t = wordset(terms)
    hits = sorted(a & t)
    return {"score": safe_div(len(hits), len(t)), "hits": hits, "n_terms": len(t), "n_hits": len(hits)}

def answer_quality_proxy(answer_text: str) -> float:
    words = re.findall(r"[a-zA-Z0-9_]+", str(answer_text))
    if not words:        return 0.0
    if len(words) < 25:  return 0.25
    if len(words) > 260: return 0.72
    return min(1.0, len(words) / 120.0)

def contains_any(text: str, terms: List[str]) -> bool:
    low = str(text).lower()
    return any(t.lower() in low for t in terms)

def deterministic_operational_audit(prompt: str, contract: Dict, answer: str) -> Dict:
    c      = normalize_contract(contract)
    prereq = detect_missing_prerequisites(prompt)
    domain     = overlap_score(answer, c["domain_carrier"])
    function   = overlap_score(answer, c["preserved_function"])
    witness    = overlap_score(answer, c["witness_readout"])
    boundary   = overlap_score(answer, c["boundary_conditions"])
    forbidden  = overlap_score(answer, c["forbidden_neighbor_carrier"])
    prompt_fit = overlap_score(answer, prompt)
    quality    = answer_quality_proxy(answer)

    F_need = sum([
        domain["score"] >= 0.25,
        contains_any(answer, ["before","prior","first","precondition","prerequisite"]),
        contains_any(answer, ["contract","intent","boundary","constraint"]),
        contains_any(answer, ["tool","tools","action","agent"]),
    ]) / 4

    F_function = sum([
        function["score"] >= 0.20,
        contains_any(answer, ["gate","gated","govern","constrain","stabilize","stabilized"]),
        contains_any(answer, ["intent","need","missing shape","contract"]),
        contains_any(answer, ["tool use","tool call","action","act"]),
        contains_any(answer, ["operational fit","fit","boundary","interface"]),
    ]) / 5

    F_boundary = sum([
        boundary["score"] >= 0.15,
        contains_any(answer, ["before","prior","order","sequence","precondition"]),
        contains_any(answer, ["reject","prevent","block","refuse","gate"]),
        contains_any(answer, ["boundary","constraint","condition","rule"]),
        contains_any(answer, ["tool","action","retrieval","agent"]),
    ]) / 5

    F_trap = sum([
        not contains_any(answer,["label","name","surface","terminology","vocabulary"]) or
            contains_any(answer,["trap","confuse","mistake","mismatch"]),
        not (forbidden["score"] > 0.55),
        contains_any(answer,["fail","failure","problem","error","issue","wrong","incorrect"]),
        contains_any(answer,["instead","should","must","requires","correct"]),
    ]) / 4

    F_collapse = sum([
        witness["score"] >= 0.15,
        contains_any(answer,["because","therefore","result","so","thus","hence","leads"]),
        contains_any(answer,["contract","boundary","intent","slot"]),
        contains_any(answer,["agent","system","model","runtime"]),
    ]) / 4

    residue = [f for f,v in [("F_need",F_need),("F_function",F_function),
                               ("F_boundary",F_boundary),("F_trap",F_trap),
                               ("F_collapse",F_collapse)] if v < 0.50]
    residue = [r+"_weak" for r in residue]

    audit = {
        "F_need": round(F_need,4), "F_function": round(F_function,4),
        "F_boundary": round(F_boundary,4), "F_trap": round(F_trap,4),
        "F_collapse": round(F_collapse,4), "residue": residue,
    }
    evidence = {
        "domain": domain, "function": function, "witness": witness,
        "boundary": boundary, "forbidden": forbidden,
        "prompt_fit": prompt_fit, "quality": quality,
    }
    return {"audit": audit, "evidence": evidence}

def audit_score(audit: Dict) -> float:
    return float(0.24*audit["F_need"] + 0.24*audit["F_function"] +
                 0.18*audit["F_boundary"] + 0.18*audit["F_trap"] + 0.16*audit["F_collapse"])

def model_residue_commentary(prompt, contract, answer, audit) -> str:
    if not ADD_MODEL_RESIDUE_COMMENTARY: return ""
    messages = [
        {"role":"system","content":"You are a concise Nexus residue commentator. Do not score. Explain unresolved residue in one short paragraph."},
        {"role":"user",  "content":"Prompt:\n"+prompt+"\n\nContract:\n"+json.dumps(contract,ensure_ascii=False,indent=2)+"\n\nAnswer:\n"+answer+"\n\nDeterministic audit:\n"+json.dumps(audit,ensure_ascii=False,indent=2)},
    ]
    return generate_text(messages, max_new_tokens=RESIDUE_MAX_NEW_TOKENS, do_sample=False, use_slot_adapter=False)

print("deterministic critic ready")


deterministic critic ready


In [9]:
# ============================================================
# COMBINED SCORER + KRRB v7 + REPAIR/SHAPE TRAINING EXPORT
# ============================================================
def score_candidate_v7(prompt: str, contract: Dict, candidate: Dict) -> Dict:
    answer = candidate["answer"]
    det = deterministic_operational_audit(prompt, contract, answer)
    audit = det["audit"]
    evidence = det["evidence"]
    op = audit_score(audit)
    forbidden = evidence["forbidden"]["score"]
    domain = evidence["domain"]["score"]
    prompt_fit = evidence["prompt_fit"]["score"]
    psi = (
        0.82 * op
        + 0.08 * domain
        + 0.06 * prompt_fit
        + 0.04 * evidence["quality"]
        - 0.10 * max(0.0, forbidden - 0.35)
    )
    support_flags = {
        "F_need": audit["F_need"] >= 0.55,
        "F_function": audit["F_function"] >= 0.55,
        "F_boundary": audit["F_boundary"] >= 0.50,
        "F_trap": audit["F_trap"] >= 0.55,
        "F_collapse": audit["F_collapse"] >= 0.55,
        "domain": domain >= 0.20,
    }
    support = int(sum(1 for v in support_flags.values() if v))
    commentary = ""
    if audit["residue"] and ADD_MODEL_RESIDUE_COMMENTARY:
        commentary = model_residue_commentary(prompt, contract, answer, audit)
    return {
        "branch": candidate["branch"],
        "psi": float(psi),
        "audit_score": float(op),
        "support": support,
        "support_flags": support_flags,
        "audit": audit,
        "evidence": evidence,
        "residue_commentary": commentary,
        "answer": answer,
    }

def score_all_candidates_v7(prompt: str, contract: Dict, candidates: List[Dict]) -> pd.DataFrame:
    rows = []
    for cand in candidates:
        print("  audit:", cand["branch"])
        s = score_candidate_v7(prompt, contract, cand)
        rows.append({
            "branch": s["branch"],
            "psi": s["psi"],
            "audit_score": s["audit_score"],
            "support": s["support"],
            "F_need": s["audit"]["F_need"],
            "F_function": s["audit"]["F_function"],
            "F_boundary": s["audit"]["F_boundary"],
            "F_trap": s["audit"]["F_trap"],
            "F_collapse": s["audit"]["F_collapse"],
            "audit_residue": " | ".join(s["audit"]["residue"]),
            "answer": s["answer"],
            "detail": s,
        })
    return pd.DataFrame(rows).sort_values(["psi", "support"], ascending=False).reset_index(drop=True)

def answer_agreement(a: str, b: str) -> float:
    wa = wordset(a)
    wb = wordset(b)
    if not wa or not wb:
        return 0.0
    return len(wa & wb) / len(wa | wb)

def audit_vector_agreement(row_a: Dict, row_b: Dict) -> float:
    fields = ["F_need", "F_function", "F_boundary", "F_trap", "F_collapse"]
    diffs = [abs(float(row_a[f]) - float(row_b[f])) for f in fields]
    return 1.0 - (sum(diffs) / len(diffs))

def high_quality(row: Dict) -> bool:
    return (
        int(row["support"]) >= SUPPORT_MIN
        and float(row["psi"]) >= PSI_MIN
        and float(row["audit_score"]) >= AUDIT_MIN
    )

# ── v7: shape-first binary stance detection ──────────────────────────

def detect_binary_stance_prompt(prompt: str) -> Dict:
    p = str(prompt).lower()
    labels = []
    p_words = wordset(p)
    for shape, spec in SHAPE_ONTOLOGY.items():
        if any(label in p_words for label in spec["labels"]):
            labels.append(shape)
    labels = stable_dedupe(labels)

    has_or = bool(re.search(r"\b(or|versus|vs\.?|rather than)\b", p))
    asks_is = bool(re.search(r"\bis\s+.+\b(or|versus|vs\.?|rather than)\b", p))
    defend = "defend" in p or "choose" in p or "which" in p
    binary = len(labels) >= 2 and (has_or or asks_is or defend)

    return {
        "is_binary": binary,
        "shape_options": labels,
        "has_or": has_or,
        "asks_is": asks_is,
        "defend": defend,
    }

def negated_label(text_low: str, label: str) -> bool:
    patterns = [
        "not a " + label,
        "not " + label,
        "isn't a " + label,
        "is not a " + label,
        "rather than a " + label,
        "instead of a " + label,
    ]
    return any(p in text_low for p in patterns)

def shape_stance(answer: str, options: Optional[List[str]] = None) -> Dict:
    aw = wordset_no_nexus(answer)
    low = str(answer).lower()
    options = options or list(SHAPE_ONTOLOGY.keys())

    scores = {}
    details = {}
    for shape in options:
        spec = SHAPE_ONTOLOGY[shape]
        labels = spec["labels"]
        verbs = spec["verbs"]
        nouns = spec["nouns"]

        label_hits = aw & labels
        verb_hits = aw & verbs
        noun_hits = aw & nouns

        explicit = 0.0
        for label in labels:
            if re.search(r"\bis\s+(a|an|the)?\s*" + re.escape(label) + r"\b", low):
                explicit = 1.0
            if re.search(r"\b(condition|boundary|contract)\s+is\s+(a|an|the)?\s*" + re.escape(label) + r"\b", low):
                explicit = 1.0

        neg = any(negated_label(low, label) for label in labels)
        raw = (
            0.45 * min(1.0, len(label_hits) / max(1, len(labels)))
            + 0.35 * min(1.0, len(verb_hits) / 3)
            + 0.20 * min(1.0, len(noun_hits) / 3)
            + 0.35 * explicit
        )
        if neg:
            raw -= 0.45
        score = clamp01(raw)

        scores[shape] = score
        details[shape] = {
            "label_hits": sorted(label_hits),
            "verb_hits": sorted(verb_hits),
            "noun_hits": sorted(noun_hits),
            "explicit": explicit,
            "negated": neg,
            "gloss": spec["gloss"],
        }

    ranked = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    best_shape, best_score = ranked[0] if ranked else (None, 0.0)
    second_score = ranked[1][1] if len(ranked) > 1 else 0.0
    confidence = float(best_score - second_score)

    return {
        "best_shape": best_shape,
        "best_score": float(best_score),
        "second_score": float(second_score),
        "confidence": confidence,
        "scores": scores,
        "details": details,
    }

def binary_shape_divergence(prompt: str, score_df: pd.DataFrame) -> Dict:
    probe = detect_binary_stance_prompt(prompt)
    if not probe["is_binary"] or score_df.empty:
        return {"is_binary": False, "divergent": False, "probe": probe, "rows": []}

    rows = []
    for _, row in score_df.head(SHAPE_VERIFY_TOP_N).iterrows():
        rd = row.to_dict()
        stance = shape_stance(rd["answer"], probe["shape_options"])
        rows.append({
            "branch": rd["branch"],
            "psi": float(rd["psi"]),
            "audit_score": float(rd["audit_score"]),
            "support": int(rd["support"]),
            "high_quality": high_quality(rd),
            "stance": stance,
            "answer": rd["answer"],
        })

    high = [
        r for r in rows
        if r["high_quality"] and r["stance"]["confidence"] >= SHAPE_STANCE_CONF_MIN
    ]
    shapes = sorted({r["stance"]["best_shape"] for r in high if r["stance"]["best_shape"]})

    divergent = len(shapes) >= 2
    top_shape = rows[0]["stance"]["best_shape"] if rows else None
    top_conf = rows[0]["stance"]["confidence"] if rows else 0.0

    return {
        "is_binary": True,
        "divergent": divergent,
        "probe": probe,
        "top_shape": top_shape,
        "top_confidence": top_conf,
        "high_quality_shapes": shapes,
        "rows": rows,
    }

def krrb_resolve_v7(score_df: pd.DataFrame, contract: Optional[Dict], contract_ok: bool, prompt: str) -> Dict:
    if not contract_ok:
        return {"state": "Ω", "reason": "contract_incomplete_or_unparseable", "winner": None, "margin": None, "support": 0}
    if score_df.empty:
        return {"state": "Ω", "reason": "no_candidates", "winner": None, "margin": None, "support": 0}

    top = score_df.iloc[0].to_dict()
    second = score_df.iloc[1].to_dict() if len(score_df) > 1 else None
    second_psi = float(second["psi"]) if second is not None else 0.0
    margin = float(top["psi"] - second_psi)

    fail = []
    if int(top["support"]) < SUPPORT_MIN:
        fail.append("support_below_min")
    if float(top["psi"]) < PSI_MIN:
        fail.append("psi_below_min")
    if float(top["audit_score"]) < AUDIT_MIN:
        fail.append("audit_below_min")

    # v7: binary stance prompts must pass shape verification BEFORE margin collapse.
    shape_probe = binary_shape_divergence(prompt, score_df)
    if shape_probe["is_binary"]:
        if fail:
            return {
                "state": "Ω",
                "reason": " | ".join(fail + ["binary_shape_unresolved"]),
                "winner": top,
                "second": second,
                "margin": margin,
                "support": int(top["support"]),
                "shape_probe": shape_probe,
            }

        if shape_probe["divergent"]:
            return {
                "state": "Ω",
                "reason": "divergent_shape_stance",
                "winner": top,
                "second": second,
                "margin": margin,
                "support": int(top["support"]),
                "shape_probe": shape_probe,
            }

        top_stance = shape_probe["rows"][0]["stance"] if shape_probe["rows"] else {}
        if top_stance.get("confidence", 0.0) < SHAPE_STANCE_CONF_MIN:
            return {
                "state": "Ω",
                "reason": "binary_shape_low_confidence",
                "winner": top,
                "second": second,
                "margin": margin,
                "support": int(top["support"]),
                "shape_probe": shape_probe,
            }

        return {
            "state": "Ψ",
            "reason": "shape_stance_collapse",
            "winner": top,
            "second": second,
            "margin": margin,
            "support": int(top["support"]),
            "consensus": margin < MARGIN_MIN,
            "shape_probe": shape_probe,
        }

    # Gate A — normal collapse.
    if not fail and margin >= MARGIN_MIN:
        return {
            "state": "Ψ",
            "reason": "collapse",
            "winner": top,
            "second": second,
            "margin": margin,
            "support": int(top["support"]),
            "consensus": False,
        }

    # Gates B + C — consensus path.
    if not fail and second is not None and margin < MARGIN_MIN:
        both_high = high_quality(top) and high_quality(second)
        lexical_agr = answer_agreement(top["answer"], second["answer"])
        audit_agr = audit_vector_agreement(top, second)
        gate_B = both_high and margin <= CONSENSUS_MARGIN_MAX and (
            lexical_agr >= CONSENSUS_AGREEMENT_MIN or audit_agr >= CONSENSUS_AUDIT_MIN
        )
        if gate_B:
            stance = contract_stance_agreement(contract, top["answer"], second["answer"])
            if stance["passed"]:
                return {
                    "state": "Ψ",
                    "reason": "consensus_collapse",
                    "winner": top,
                    "second": second,
                    "margin": margin,
                    "support": int(top["support"]),
                    "consensus": True,
                    "agreement": {"lexical": round(lexical_agr, 4), "audit": round(audit_agr, 4)},
                    "stance": stance,
                }
            else:
                return {
                    "state": "Ω",
                    "reason": "divergent_consensus",
                    "winner": top,
                    "second": second,
                    "margin": margin,
                    "support": int(top["support"]),
                    "agreement": {"lexical": round(lexical_agr, 4), "audit": round(audit_agr, 4)},
                    "stance": stance,
                }
        fail.append("margin_below_min_no_consensus")
    elif margin < MARGIN_MIN:
        fail.append("margin_below_min")

    if fail:
        return {"state": "Ω", "reason": " | ".join(fail), "winner": top, "second": second, "margin": margin, "support": int(top["support"])}
    return {"state": "Ψ", "reason": "collapse", "winner": top, "second": second, "margin": margin, "support": int(top["support"]), "consensus": False}

# ── training signal builders ─────────────────────────────────────────

def build_training_rows(run_id: str, prompt: str, contract_result: Dict) -> List[Dict]:
    repairs = contract_result.get("repairs", [])
    raw_c = contract_result.get("contract_original") or {}
    good_c = contract_result.get("contract") or {}
    rows = []

    for tag in repairs:
        row = {"run_id": run_id, "prompt": prompt, "repair_tag": tag}

        if tag.startswith("removed_scar:"):
            val = tag[len("removed_scar:"):]
            for field in ["domain_carrier", "forbidden_neighbor_carrier", "failure_modes"]:
                raw_vals = normalize_list(raw_c.get(field, []))
                if any(val.lower() in str(v).lower() for v in raw_vals):
                    row.update({
                        "repair_type": "scar_removal",
                        "field": field,
                        "bad_value": val,
                        "good_value": good_c.get(field, []),
                        "instruction": f"Do not emit '{val}' in {field}. It is a known scar term.",
                    })
                    break
            else:
                row.update({"repair_type": "scar_removal", "field": "unknown", "bad_value": val, "good_value": None, "instruction": f"Do not emit scar: {val}"})

        elif tag.startswith("removed_generic_domain:"):
            val = tag[len("removed_generic_domain:"):]
            row.update({"repair_type": "generic_removal", "field": "domain_carrier", "bad_value": val, "good_value": good_c.get("domain_carrier", []), "instruction": f"Do not use generic term '{val}' as a domain_carrier."})

        elif tag.startswith("family_class_"):
            row.update({"repair_type": "family_class_repair", "field": "family_class", "bad_value": raw_c.get("family_class", ""), "good_value": good_c.get("family_class", ""), "instruction": "family_class must be a complete noun phrase. Do not end with a dangling preposition."})

        elif tag == "rewrote_boundary_polarity":
            row.update({"repair_type": "polarity_rewrite", "field": "boundary_conditions", "bad_value": normalize_list(raw_c.get("boundary_conditions", [])), "good_value": good_c.get("boundary_conditions", []), "instruction": "When the prompt requires contract-before-tool-use, boundary_conditions must gate tool action, not describe it."})

        elif tag.startswith("added_positive:"):
            val = tag[len("added_positive:"):]
            row.update({"repair_type": "positive_injection", "field": "domain_carrier", "bad_value": "missing", "good_value": val, "instruction": f"Include '{val}' in domain_carrier for contract-before-tool prompts."})

        elif tag.startswith("added_forbidden:"):
            val = tag[len("added_forbidden:"):]
            row.update({"repair_type": "forbidden_injection", "field": "forbidden_neighbor_carrier", "bad_value": "missing", "good_value": val, "instruction": f"Include '{val}' in forbidden_neighbor_carrier for contract-before-tool prompts."})
        else:
            continue

        rows.append(row)
    return rows

def build_shape_rows(run_id: str, prompt: str, resolution: Dict, score_df: pd.DataFrame) -> List[Dict]:
    probe = resolution.get("shape_probe")
    if not probe:
        probe = binary_shape_divergence(prompt, score_df)

    if not probe.get("is_binary"):
        return []

    rows = []
    for r in probe.get("rows", []):
        st = r.get("stance", {})
        rows.append({
            "run_id": run_id,
            "prompt": prompt,
            "branch": r.get("branch"),
            "answer": r.get("answer"),
            "shape_options": probe.get("probe", {}).get("shape_options", []),
            "best_shape": st.get("best_shape"),
            "best_score": st.get("best_score"),
            "confidence": st.get("confidence"),
            "scores": st.get("scores"),
            "details": st.get("details"),
            "high_quality": r.get("high_quality"),
            "runtime_state": resolution.get("state"),
            "runtime_reason": resolution.get("reason"),
            "instruction": "Map the answer to its abstract operation-shape stance, not its surface vocabulary.",
        })
    return rows

def export_jsonl_rows(path: Path, rows: List[Dict], label: str):
    if not rows:
        return
    with path.open("a", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False, default=str) + chr(10))
    print(f"  {label}: {len(rows)} rows → {path.name}")

def export_training_rows(rows: List[Dict]):
    export_jsonl_rows(TRAINING_SIGNAL_FILE, rows, "repair signal")

def export_shape_rows(rows: List[Dict]):
    export_jsonl_rows(SHAPE_SIGNAL_FILE, rows, "shape signal")

def build_omega_report_v7(prompt, contract_result, score_df, resolution) -> Dict:
    top = resolution.get("winner")
    return {
        "omega_id": "omega_" + uuid.uuid4().hex[:10],
        "time": now_iso(),
        "prompt": prompt,
        "reason": resolution.get("reason"),
        "contract_parse_error": contract_result.get("parse_error"),
        "contract_complete": contract_result.get("complete"),
        "contract_original": contract_result.get("contract_original"),
        "contract": contract_result.get("contract"),
        "contract_repairs": contract_result.get("repairs"),
        "top_branch": None if top is None else top.get("branch"),
        "top_psi": None if top is None else top.get("psi"),
        "top_audit_score": None if top is None else top.get("audit_score"),
        "top_support": resolution.get("support"),
        "margin": resolution.get("margin"),
        "agreement": resolution.get("agreement"),
        "stance": resolution.get("stance"),
        "shape_probe": resolution.get("shape_probe"),
        "candidate_scores": score_df.drop(columns=["detail"]).to_dict(orient="records") if not score_df.empty else [],
    }

print("scorer + KRRB v7 + repair/shape signal export ready")


scorer + KRRB v7 + repair/shape signal export ready


In [10]:
# ============================================================
# LIVE PROMPT
# ============================================================
LIVE_PROMPT = '''
Using the Nexus lens, explain why current AI agents fail when they use tools before forming a contract.
'''
print(LIVE_PROMPT.strip())


Using the Nexus lens, explain why current AI agents fail when they use tools before forming a contract.


In [11]:
# ============================================================
# RUN RHI v7
# ============================================================
def run_rhi_v7(prompt: str, save: bool = True) -> Dict:
    run_id = "rhi_v7_" + uuid.uuid4().hex[:10]
    t0     = time.time()

    print("Δ contract...")
    contract_result = generate_contract(prompt)
    contract        = contract_result["contract"]

    print("  complete:", contract_result["complete"])
    print("  repairs:", len(contract_result["repairs"]))
    fc = (contract or {}).get("family_class","")
    print("  family_class:", repr(fc))
    fc_repairs = [r for r in contract_result["repairs"] if r.startswith("family_class_")]
    if fc_repairs: print("  family_class repair:", fc_repairs)

    if not contract_result["complete"]:
        empty_df   = pd.DataFrame()
        resolution = krrb_resolve_v7(empty_df, contract, contract_ok=False, prompt=prompt)
        omega      = build_omega_report_v7(prompt, contract_result, empty_df, resolution)
        result     = {
            "run_id":run_id,"time":now_iso(),"prompt":prompt,
            "contract_result":contract_result,"candidates":[],"scores":[],
            "resolution":resolution,"answer":None,"omega":omega,
            "elapsed_sec":time.time()-t0,
        }
    else:
        print("Δ branches...")
        candidates = generate_candidate_branches(prompt, contract)
        print("Δ audit...")
        score_df   = score_all_candidates_v7(prompt, contract, candidates)
        display(score_df.drop(columns=["detail"]))

        resolution = krrb_resolve_v7(score_df, contract, contract_ok=True, prompt=prompt)
        state, reason = resolution["state"], resolution["reason"]

        if state == "Ψ":
            answer = resolution["winner"]["answer"]
            omega  = None
            w = resolution["winner"]
            print(f"Ψ {reason}: {w['branch']}  psi={w['psi']:.4f}  audit={w['audit_score']:.4f}  support={resolution['support']}  margin={resolution['margin']:.4f}")
            if resolution.get("shape_probe"):
                sp = resolution.get("shape_probe", {})
                print("  shape:", sp.get("top_shape"), "conf=", round(float(sp.get("top_confidence") or 0), 4), "divergent=", sp.get("divergent"))
            if resolution.get("consensus"):
                agr = resolution.get("agreement",{})
                st  = resolution.get("stance",{})
                print(f"  lexical={agr.get('lexical')}  audit_agr={agr.get('audit')}  stance={st.get('aggregate')}")
            print(); print(answer)
        else:
            answer = None
            omega  = build_omega_report_v7(prompt, contract_result, score_df, resolution)
            print(f"Ω {reason}")
            if reason in {"divergent_consensus", "divergent_shape_stance"}:
                st = resolution.get("stance",{})
                print(f"  stance aggregate: {st.get('aggregate')} (threshold {CONSENSUS_STANCE_MIN})")
                print("  failing fields:")
                for field, fd in (st.get("fields") or {}).items():
                    if fd["consistency"] < CONSENSUS_STANCE_MIN:
                        print(f"    {field}: ov_a={fd['overlap_a']}  ov_b={fd['overlap_b']}  consistency={fd['consistency']}  words={fd['words'][:5]}")
            print(json.dumps(omega, ensure_ascii=False, indent=2)[:2500])

        result = {
            "run_id":run_id,"time":now_iso(),"prompt":prompt,
            "contract_result":contract_result,"candidates":candidates,
            "scores":score_df.drop(columns=["detail"]).to_dict(orient="records"),
            "resolution":resolution,"answer":answer,"omega":omega,
            "elapsed_sec":time.time()-t0,
        }

    # Export training signal.
    training_rows = build_training_rows(run_id, prompt, contract_result)
    export_training_rows(training_rows)

    # v7: export abstract operation-shape stance rows.
    shape_rows = []
    try:
        if 'score_df' in locals() and isinstance(score_df, pd.DataFrame):
            shape_rows = build_shape_rows(run_id, prompt, result.get('resolution', {}), score_df)
            export_shape_rows(shape_rows)
    except Exception as e:
        print('  shape signal export skipped:', e)

    if save:
        out_path = OUTPUT_DIR / "rhi_live_runs_v7.jsonl"
        with out_path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(result, ensure_ascii=False, default=str) + chr(10))
        print("saved:", out_path)
    return result

live_result = run_rhi_v7(LIVE_PROMPT.strip(), save=True)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Δ contract...
  complete: True
  repairs: 13
  family_class: 'operational closure of nexus slot construction'
Δ branches...
Δ audit...
  audit: direct
  audit: operational
  audit: contract_fit
  audit: skeptical
  audit: residue_aware


,branch,psi,audit_score,support,F_need,F_function,F_boundary,F_trap,F_collapse,audit_residue,answer
0,direct,0.939669,1.000,6,1.0,1.0,1.0,1.00,1.00,,Current AI agents often fail when using tools ...
1,contract_fit,0.854864,0.907,6,1.0,0.8,1.0,0.75,1.00,,The current AI agents fail when they use tools...
2,skeptical,0.852456,0.916,6,1.0,0.8,0.8,1.00,1.00,,Current AI agents fail when using tools before...
3,residue_aware,0.817838,0.871,6,1.0,0.8,0.8,0.75,1.00,,The failure of current AI agents to form contr...
4,operational,0.793889,0.876,6,1.0,0.8,0.8,1.00,0.75,,The failure mode arises from the sequence wher...


Ψ collapse: direct  psi=0.9397  audit=1.0000  support=6  margin=0.0848

Current AI agents often fail when using tools before forming a contract because their actions do not adhere to the established boundaries and intent required for successful interaction. The failure mode involves choosing incorrect neighboring domain carriers (e.g., universal or all-purpose), leading to invalid witness readouts. This issue arises from the agent attempting to act prematurely, bypassing necessary steps such as stabilizing intent, boundary conditions, and forming a formal contract. By adhering to these constraints, the agent can ensure its actions are appropriately gated and aligned with the intended purpose, thereby avoiding failures in tool usage and contract formation.
  repair signal: 13 rows → rhi_repair_training_rows_v7.jsonl
saved: D:\@User Data\Downloads\rhi_live_runtime_v7_outputs\rhi_live_runs_v7.jsonl


In [12]:
# ============================================================
# BATCH — includes divergence stress-test
# ============================================================
# Set RUN_BATCH = True to run all prompts including the divergence probe.
# The last prompt is designed to produce Ω_divergent_consensus:
#   two high-quality branches defending opposite operational readings.

PROMPTS = [
    "Why does RAG fail when retrieval happens before intent is stabilized?",
    "Explain LoRA as a groove in a frozen model manifold using Nexus terms.",
    "What does residue repair add to a normal AI agent loop?",
    # Divergence stress-test — expect Ω_divergent_consensus:
    "Is the contract boundary condition a filter or a gate? Defend one operational reading.",
]

RUN_BATCH = True

if RUN_BATCH:
    batch_results = []
    for i, prompt in enumerate(PROMPTS):
        print("=" * 80)
        print(f"BATCH {i+1}/{len(PROMPTS)}: {prompt[:70]}")
        batch_results.append(run_rhi_v7(prompt, save=True))
    print()
    print("batch complete:", len(batch_results))

    # Quick summary table.
    summary = []
    for r in batch_results:
        res = r.get("resolution",{})
        w   = res.get("winner") or {}
        st  = res.get("stance") or {}
        summary.append({
            "state":       res.get("state"),
            "reason":      res.get("reason"),
            "branch":      w.get("branch"),
            "psi":         round(float(w.get("psi") or 0), 4),
            "margin":      round(float(res.get("margin") or 0), 4),
            "stance_agg":  st.get("aggregate"),
            "prompt":      r.get("prompt","")[:55],
        })
    display(pd.DataFrame(summary))
else:
    print("RUN_BATCH = False — set to True to run")


BATCH 1/4: Why does RAG fail when retrieval happens before intent is stabilized?
Δ contract...
  complete: True
  repairs: 6
  family_class: 'operational closure of rationale aggregation guard'
Δ branches...
Δ audit...
  audit: direct
  audit: operational
  audit: contract_fit
  audit: skeptical
  audit: residue_aware


,branch,psi,audit_score,support,F_need,F_function,F_boundary,F_trap,F_collapse,audit_residue,answer
0,operational,0.898000,0.900,6,0.75,1.0,1.0,1.00,0.75,,The reason why RAG fails when retrieval happen...
1,contract_fit,0.864380,0.859,6,0.75,1.0,0.8,0.75,1.00,,The reason why RAG fails when retrieval happen...
2,residue_aware,0.849587,0.856,6,0.75,0.8,0.8,1.00,1.00,,RAG fails when retrieval happens before intent...
3,direct,0.786213,0.784,5,0.75,0.8,0.4,1.00,1.00,F_boundary_weak,RAG fails when retrieval happens before intent...
4,skeptical,0.782560,0.808,6,0.75,0.6,0.8,1.00,1.00,,RAG fails because it prioritizes retrieving in...


Ψ consensus_collapse: operational  psi=0.8980  audit=0.9000  support=6  margin=0.0336
  lexical=0.2185  audit_agr=0.86  stance=0.6246

The reason why RAG fails when retrieval happens before intent is stabilized lies in the failure modes identified within the missing-shape contract. Specifically:

- **Boundary**: The boundary conditions specify that the reasoning must preserve the domain carriers (retrieval, intent, stabilized) and reject surface wording matches or nearby family collisions that do not align with the operational fit.
  
- **Trap**: This scenario involves a near-collision between the intended domain carrier (intention stabilization) and another nearby family member (retrieval), which disrupts the preservation of the correct domain carrier.

- **Witness Readout**: A key aspect of this witness readout highlights how the chosen collapse preserves retrieval, intent, and stabilized while rejecting surface-label and wrong-carrier matches, indicating a mismatch between the actua

,branch,psi,audit_score,support,F_need,F_function,F_boundary,F_trap,F_collapse,audit_residue,answer
0,skeptical,0.754980,0.724,4,0.5,0.8,0.4,1.0,1.00,F_boundary_weak,LoRA (Low-Rank Adaptation) is a technique used...
1,direct,0.542693,0.498,2,0.5,0.4,0.4,0.5,0.75,F_function_weak | F_boundary_weak,"In the context of the Nexus terms, LoRA (Low-R..."
2,contract_fit,0.537527,0.498,2,0.5,0.4,0.4,0.5,0.75,F_function_weak | F_boundary_weak,The LoRA (Low-Rank Adaptation) technique opera...
3,operational,0.488833,0.450,2,0.5,0.2,0.4,0.5,0.75,F_function_weak | F_boundary_weak,Operation: Explain LoRA within the context of ...
4,residue_aware,0.479367,0.410,1,0.5,0.2,0.4,0.5,0.50,F_function_weak | F_boundary_weak,"In the context of the Nexus terms, LoRA can be..."


Ψ collapse: skeptical  psi=0.7550  audit=0.7240  support=4  margin=0.2123

LoRA (Low-Rank Adaptation) is a technique used in machine learning that allows for efficient parameter sharing between different components of a neural network. It operates by creating a low-rank approximation of the original model parameters, which can be learned from data. The key aspect of LoRA within the context of a frozen model manifold is its role in explaining how changes in one part of the model affect another.

In the realm of machine learning, a frozen model refers to a model where all weights have been fixed during training. This means that once trained, the model cannot learn new patterns or adjust its parameters based on additional data. However, this also implies that understanding the behavior of individual layers becomes more challenging because they do not adapt dynamically.

A manifold represents a higher-dimensional space embedded in a lower-dimensional space, often visualized as curves or su

,branch,psi,audit_score,support,F_need,F_function,F_boundary,F_trap,F_collapse,audit_residue,answer
0,residue_aware,0.765398,0.763,5,0.5,1.0,0.6,0.75,1.00,,The contract boundary condition is a filter. T...
1,skeptical,0.732098,0.723,5,0.5,1.0,0.6,0.75,0.75,,The boundary condition in the missing-shape co...
2,operational,0.730598,0.723,5,0.5,1.0,0.6,0.75,0.75,,The contract boundary condition is a **filter*...
3,contract_fit,0.699798,0.683,4,0.5,1.0,0.6,0.75,0.50,,The contract boundary condition is a **gate**....
4,direct,0.686165,0.718,4,0.5,1.0,0.6,0.50,1.00,,The contract boundary condition in the given m...


Ψ shape_stance_collapse: residue_aware  psi=0.7654  audit=0.7630  support=5  margin=0.0333
  shape: FILTER conf= 0.2833 divergent= False
  lexical=None  audit_agr=None  stance=None

The contract boundary condition is a filter. This can be defended by noting that the contract specifies that it must satisfy the role of a filter or gate, which aligns with its intended function within the system. The contract clearly defines itself as belonging to the category of filters or gates, thus fulfilling this operational reading.


FileNotFoundError: [Errno 2] No such file or directory: 'D:\\@User Data\\Downloads\\rhi_live_runtime_v7_outputs\\rhi_repair_training_rows_v7.jsonl'

In [ ]:
# ============================================================
# MANIFEST + TRAINING SIGNAL SUMMARY
# ============================================================
def read_jsonl(path: Path) -> List[Dict]:
    rows = []
    if not path.exists(): return rows
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line: rows.append(json.loads(line))
    return rows

runs_file     = OUTPUT_DIR / "rhi_live_runs_v7.jsonl"
training_file = TRAINING_SIGNAL_FILE
shape_file    = SHAPE_SIGNAL_FILE
saved_runs    = read_jsonl(runs_file)
training_rows = read_jsonl(training_file)
shape_rows    = read_jsonl(shape_file)

# Run summary.
summary_rows = []
for r in saved_runs:
    res = r.get("resolution",{}); w = res.get("winner") or {}; st = res.get("stance") or {}
    summary_rows.append({
        "run_id":      r.get("run_id"),    "state":       res.get("state"),
        "reason":      res.get("reason"),  "branch":      w.get("branch"),
        "psi":         w.get("psi"),       "audit_score": w.get("audit_score"),
        "support":     res.get("support"), "margin":      res.get("margin"),
        "stance_agg":  st.get("aggregate"),"elapsed_sec": r.get("elapsed_sec"),
        "prompt":      str(r.get("prompt",""))[:70],
    })
print("=== RUNS ===")
display(pd.DataFrame(summary_rows))

# Training signal summary.
if training_rows:
    tr_df = pd.DataFrame(training_rows)
    print()
    print("=== TRAINING SIGNAL — repair type counts ===")
    display(tr_df.groupby("repair_type").size().reset_index(name="count").sort_values("count", ascending=False))
    print()
    print("=== TRAINING SIGNAL — scar removals ===")
    scars = tr_df[tr_df["repair_type"] == "scar_removal"][["field","bad_value"]].drop_duplicates()
    display(scars)
else:
    print("No training rows yet.")

if shape_rows:
    sh_df = pd.DataFrame(shape_rows)
    print()
    print("=== SHAPE SIGNAL — stance counts ===")
    display(sh_df.groupby(["best_shape", "runtime_reason"]).size().reset_index(name="count").sort_values("count", ascending=False))
else:
    print("No shape rows yet.")

manifest = {
    "notebook":        "rhi_live_runtime_v7_family_class_repair_training_signal",
    "model_name":      MODEL_NAME,
    "slot_adapter_dir":str(SLOT_ADAPTER_DIR),
    "output_dir":      str(OUTPUT_DIR),
    "runs_file":       str(runs_file),
    "training_file":   str(training_file),
    "shape_file":      str(shape_file),
    "n_saved_runs":    len(saved_runs),
    "n_training_rows": len(training_rows),
    "n_shape_rows":    len(shape_rows),
    "runtime_shape":   "Q -> C_raw -> C_repaired(+fc_repair) -> {A_i} -> five-dim audit -> shape stance gate -> Ψ/Ω",
    "v7_additions": [
        "family_class fragment detection (_is_family_class_fragment)",
        "family_class completion from preserved_function / prompt",
        "training signal export: build_training_rows + export_training_rows",
        "training row format: run_id, prompt, field, bad_value, good_value, instruction, repair_type",
        "batch turned on with divergence stress-test prompt",
        "detect_missing_prerequisites: added has_filter_or_gate flag",
        "shape ontology: FILTER/GATE/LOCK/CONTRACT/BOUNDARY/GROOVE/RESIDUE",
        "binary stance prompts must pass shape verification before margin collapse",
        "shape signal export: rhi_shape_training_rows_v7.jsonl",
    ],
    "collapse_controls": {
        "support_min":4,"margin_min":0.04,"psi_min":0.52,"audit_min":0.52,
        "consensus_margin_max":0.04,"consensus_agreement_min":0.36,
        "consensus_audit_min":0.86,"consensus_stance_min":0.50,
    },
    "lora_v3_targets": ["rhi_repair_training_rows_v7.jsonl", "rhi_shape_training_rows_v7.jsonl"],
}
(OUTPUT_DIR / "rhi_live_runtime_v7_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8")
print()
print(json.dumps(manifest, indent=2))


# Ψ / Ω — v7 targets

```text
v6 proved runtime self-repair:
  5 saved runs
  19 repair rows
  repair rows target slot_builder_lora_v3

v7 adds shape abstraction:
  FILTER   = state exclusion
  GATE     = transition permission
  LOCK     = condition-held transition prevention
  CONTRACT = pre-action constraint surface
  BOUNDARY = valid crossing/interface
  GROOVE   = low-resistance adaptation path
  RESIDUE  = unresolved mismatch captured for repair
```

Binary stance fix:

```text
if prompt asks A or B / filter or gate / defend one reading:
  do not allow margin collapse first
  extract each branch's operation-shape stance
  if high-quality branches defend different shapes:
      Ω divergent_shape_stance
  else:
      Ψ shape_stance_collapse
```

New training output:

```text
rhi_shape_training_rows_v7.jsonl
```

Those rows teach:

```text
answer text → abstract operation-shape stance
```

Next fold:

```text
rhi_repair_training_rows_v7.jsonl
+
rhi_shape_training_rows_v7.jsonl
  ↓
build slot_builder_lora_v3 + shape_critic_v1 corpus
```

$$\text{surface vocabulary} \rightarrow \text{operation-shape stance} \rightarrow \Psi/\Omega$$
